# Notebook 1: Data Collection

Pulls raw player game logs from `nba_api` and scrapes salary data from Basketball-Reference for seasons 2020-21 through 2024-25. All data is saved as CSVs in `../data/raw/`.

**Expected outputs:**
- `game_logs_YYYY-YY.csv` × 5 (~25,000 rows each)
- `salaries_YYYY-YY.csv` × 5 (~450–500 rows each)

In [5]:
import time
import re
from pathlib import Path

import polars as pl
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
from nba_api.stats.endpoints.playergamelogs import PlayerGameLogs

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

SEASONS = [
    ("2020-21", 2021),
    ("2021-22", 2022),
    ("2022-23", 2023),
    ("2023-24", 2024),
    ("2024-25", 2025),
]

print(f"Raw data directory: {RAW_DIR.resolve()}")
print(f"Seasons to collect: {[s[0] for s in SEASONS]}")

Raw data directory: /Users/allenliu/Desktop/CIS 2450/FINAL PROJECT/NBA-Salary-Prediction/data/raw
Seasons to collect: ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25']


## 1. Game Logs — nba_api

`PlayerGameLogs` pulls all player game logs for an entire season in one API call (~25k rows/season). We sleep 2 seconds before each call to stay within the NBA API rate limit (~1 req/s). Already-fetched seasons are skipped.

In [2]:
def fetch_game_logs(season_str: str) -> pl.DataFrame:
    """Fetch all player game logs for one season from nba_api."""
    time.sleep(2.0)  # Rate limit guard — NBA API throttles ~1 req/s
    logs = PlayerGameLogs(
        season_nullable=season_str,
        season_type_nullable="Regular Season",
        timeout=60,
    )
    df = pl.from_pandas(logs.get_data_frames()[0])
    return df.with_columns(pl.lit(season_str).alias("SEASON"))


for season_str, end_year in tqdm(SEASONS, desc="Fetching game logs"):
    out_path = RAW_DIR / f"game_logs_{season_str}.csv"
    if out_path.exists():
        print(f"  [SKIP] {out_path.name} already exists")
        continue

    print(f"  Fetching {season_str}...")
    for attempt in range(1, 4):  # Retry up to 3 times
        try:
            df = fetch_game_logs(season_str)
            df.write_csv(out_path)
            print(f"  Saved {out_path.name} — {len(df):,} rows")
            break
        except Exception as e:
            print(f"  Attempt {attempt} failed: {e}")
            if attempt < 3:
                time.sleep(5.0)
            else:
                print(f"  ERROR: Could not fetch {season_str} after 3 attempts")

print("\nGame log collection complete.")

Fetching game logs:   0%|          | 0/5 [00:00<?, ?it/s]

  Fetching 2020-21...


Fetching game logs:  20%|██        | 1/5 [00:02<00:11,  2.93s/it]

  Saved game_logs_2020-21.csv — 23,054 rows
  Fetching 2021-22...


Fetching game logs:  40%|████      | 2/5 [00:05<00:08,  2.82s/it]

...

  Saved game_logs_2022-23.csv — 25,894 rows
  Fetching 2023-24...


Fetching game logs:  80%|████████  | 4/5 [00:11<00:02,  2.90s/it]

  Saved game_logs_2023-24.csv — 26,401 rows
  Fetching 2024-25...


Fetching game logs: 100%|██████████| 5/5 [00:14<00:00,  2.88s/it]

  Saved game_logs_2024-25.csv — 26,306 rows

Game log collection complete.


**Verify game log files:**

In [3]:
for season_str, _ in SEASONS:
    path = RAW_DIR / f"game_logs_{season_str}.csv"
    if path.exists():
        df = pl.read_csv(path)
        print(f"game_logs_{season_str}.csv: {len(df):,} rows × {df.width} cols")
    else:
        print(f"MISSING: game_logs_{season_str}.csv")

game_logs_2020-21.csv: 23,054 rows × 71 cols
game_logs_2021-22.csv: 26,039 rows × 71 cols
game_logs_2022-23.csv: 25,894 rows × 71 cols
game_logs_2023-24.csv: 26,401 rows × 71 cols
game_logs_2024-25.csv: 26,306 rows × 71 cols


## 2. Salaries — Basketball-Reference (Team Pages)

Scrapes the salary table (`#salaries2`) from each team's roster page for each season. BBRef's scraping policy allows up to 20 requests/minute — we sleep 5s between each request (= 12 req/min).

- **URL format:** `https://www.basketball-reference.com/teams/{TEAM}/{END_YEAR}.html`
- **30 teams × 5 seasons = 150 pages** (~12.5 min total)
- **Traded players** appear on multiple team pages with the same salary — deduplicated by `BBREF_ID`
- **Bonus:** BBRef player ID extracted from href (e.g. `curryst01`) — used for reliable joining in Notebook 2

In [6]:
BBREF_TEAMS = [
    "ATL", "BOS", "BRK", "CHO", "CHI", "CLE", "DAL", "DEN", "DET", "GSW",
    "HOU", "IND", "LAC", "LAL", "MEM", "MIA", "MIL", "MIN", "NOP", "NYK",
    "OKC", "ORL", "PHI", "PHO", "POR", "SAC", "SAS", "TOR", "UTA", "WAS",
]

BBREF_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Referer": "https://www.basketball-reference.com/",
    "Accept-Language": "en-US,en;q=0.9",
}


def scrape_team_salaries(team: str, end_year: int) -> list:
    """Scrape salary table from one BBRef team page. Returns list of dicts."""
    url = f"https://www.basketball-reference.com/teams/{team}/{end_year}.html"
    resp = requests.get(url, headers=BBREF_HEADERS, timeout=30)

    if resp.status_code == 429:
        raise RuntimeError(f"BBRef rate limit (429) hit — wait a few minutes and retry.")
    if resp.status_code == 404:
        print(f"    404 for {team} {end_year} — skipping")
        return []
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "lxml")
    table = soup.find("table", {"id": "salaries2"})

    # BBRef sometimes hides tables inside HTML comments
    if table is None:
        from bs4 import Comment
        for comment in soup.find_all(string=lambda t: isinstance(t, Comment)):
            if 'id="salaries2"' in comment:
                table = BeautifulSoup(comment, "lxml").find("table", {"id": "salaries2"})
                break

    if table is None:
        print(f"    No salary table for {team} {end_year}")
        return []

    rows = []
    for tr in table.find("tbody").find_all("tr"):
        player_td = tr.find("td", {"data-stat": "player"})
        salary_td = tr.find("td", {"data-stat": "salary"})
        if not player_td or not salary_td:
            continue

        player_name = player_td.get_text(strip=True)
        if not player_name:
            continue

        # Extract BBRef player ID from href: /players/c/curryst01.html → curryst01
        a_tag = player_td.find("a")
        bbref_id = a_tag["href"].split("/")[-1].replace(".html", "") if a_tag else ""

        salary_raw = re.sub(r"[\$,]", "", salary_td.get_text(strip=True)).strip()
        if not salary_raw or not salary_raw.isdigit():
            continue

        rows.append({
            "PLAYER_NAME": player_name,
            "BBREF_ID": bbref_id,
            "TEAM": team,
            "SALARY": float(salary_raw),
        })

    return rows


print(f"Defined scrape_team_salaries() — {len(BBREF_TEAMS)} teams configured.")

Defined scrape_team_salaries() — 30 teams configured.


In [7]:
for season_str, end_year in tqdm(SEASONS, desc="Seasons"):
    out_path = RAW_DIR / f"salaries_{season_str}.csv"
    if out_path.exists():
        print(f"  [SKIP] {out_path.name} already exists")
        continue

    print(f"\n  Season {season_str} (end year {end_year}) — scraping 30 teams...")
    all_rows = []

    for team in BBREF_TEAMS:
        try:
            rows = scrape_team_salaries(team, end_year)
            all_rows.extend(rows)
        except RuntimeError as e:
            # Rate limit — stop immediately so user can retry
            raise
        except Exception as e:
            print(f"    ERROR {team}: {e}")

        time.sleep(5.0)  # 5s between requests = 12 req/min (limit: 20 req/min)

    if not all_rows:
        print(f"  WARNING: No salary data collected for {season_str}")
        continue

    df = pl.DataFrame(all_rows)

    # Deduplicate: traded players appear on 2+ team pages with the same salary
    # Prefer dedup by BBREF_ID (reliable); fall back to PLAYER_NAME for rows without ID
    df_with_id = df.filter(pl.col("BBREF_ID") != "").unique(subset=["BBREF_ID"], keep="first")
    df_no_id = df.filter(pl.col("BBREF_ID") == "").unique(subset=["PLAYER_NAME"], keep="first")
    df = pl.concat([df_with_id, df_no_id], how="diagonal_relaxed")

    df = df.with_columns(pl.lit(season_str).alias("SEASON"))
    df.write_csv(out_path)
    print(f"  Saved {out_path.name} — {len(df):,} unique players")

print("\nSalary collection complete.")

Seasons:   0%|          | 0/5 [00:00<?, ?it/s]


  Season 2020-21 (end year 2021) — scraping 30 teams...


Seasons:  20%|██        | 1/5 [03:29<13:57, 209.34s/it]

  Saved salaries_2020-21.csv — 574 unique players

  Season 2021-22 (end year 2022) — scraping 30 teams...


Seasons:  40%|████      | 2/5 [06:08<09:00, 180.09s/it]

...

  Saved salaries_2022-23.csv — 511 unique players

  Season 2023-24 (end year 2024) — scraping 30 teams...


Seasons:  80%|████████  | 4/5 [11:28<02:46, 166.48s/it]

  Saved salaries_2023-24.csv — 531 unique players

  Season 2024-25 (end year 2025) — scraping 30 teams...


Seasons: 100%|██████████| 5/5 [14:51<00:00, 178.27s/it]

  Saved salaries_2024-25.csv — 518 unique players

Salary collection complete.


**Verify salary files:**

In [8]:
for season_str, _ in SEASONS:
    path = RAW_DIR / f"salaries_{season_str}.csv"
    if path.exists():
        df = pl.read_csv(path)
        print(f"salaries_{season_str}.csv: {len(df):,} rows")
    else:
        print(f"MISSING: salaries_{season_str}.csv")

salaries_2020-21.csv: 574 rows
salaries_2021-22.csv: 657 rows
salaries_2022-23.csv: 511 rows
salaries_2023-24.csv: 531 rows
salaries_2024-25.csv: 518 rows


## 3. Player Bios — Basketball-Reference

Scrapes the roster table (`#roster`) from each team's BBRef page for the 2024-25 season — the same pages already used for salaries, so no new URLs needed. Collects position, birth date, height, weight, years of experience, and college for all 30 teams in one pass (~2.5 min). Results saved to `player_bios.csv` and joined in Notebook 2 to compute age-at-game-time and years of experience per game row.

In [10]:
bio_out = RAW_DIR / "player_bios.csv"

BBREF_TEAMS = [
    "ATL", "BOS", "BRK", "CHO", "CHI", "CLE", "DAL", "DEN", "DET", "GSW",
    "HOU", "IND", "LAC", "LAL", "MEM", "MIA", "MIL", "MIN", "NOP", "NYK",
    "OKC", "ORL", "PHI", "PHO", "POR", "SAC", "SAS", "TOR", "UTA", "WAS",
]

BBREF_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Referer": "https://www.basketball-reference.com/",
    "Accept-Language": "en-US,en;q=0.9",
}

if bio_out.exists():
    print(f"[SKIP] {bio_out.name} already exists")
else:
    # Scrape roster table from one season only (2024-25) — bio data doesn't change year to year
    # Same team URLs we already used for salaries, no extra rate limit cost
    SCRAPE_YEAR = 2025
    all_rows = []

    for team in tqdm(BBREF_TEAMS, desc="Scraping rosters"):
        url = f"https://www.basketball-reference.com/teams/{team}/{SCRAPE_YEAR}.html"
        try:
            resp = requests.get(url, headers=BBREF_HEADERS, timeout=30)
            if resp.status_code == 429:
                raise RuntimeError("BBRef rate limit (429) hit — wait and retry.")
            if resp.status_code == 404:
                print(f"  404 for {team} — skipping")
                time.sleep(5.0)
                continue
            resp.raise_for_status()

            soup = BeautifulSoup(resp.text, "lxml")
            table = soup.find("table", {"id": "roster"})

            # BBRef sometimes hides tables in HTML comments
            if table is None:
                from bs4 import Comment
                for comment in soup.find_all(string=lambda t: isinstance(t, Comment)):
                    if 'id="roster"' in comment:
                        table = BeautifulSoup(comment, "lxml").find("table", {"id": "roster"})
                        break

            if table is None:
                print(f"  No roster table for {team}")
                time.sleep(5.0)
                continue

            for tr in table.find("tbody").find_all("tr"):
                player_td = tr.find("td", {"data-stat": "player"})
                if not player_td:
                    continue
                player_name = player_td.get_text(strip=True)
                if not player_name:
                    continue

                a_tag = player_td.find("a")
                bbref_id = a_tag["href"].split("/")[-1].replace(".html", "") if a_tag else ""

                def get(stat):
                    td = tr.find("td", {"data-stat": stat})
                    return td.get_text(strip=True) if td else ""

                all_rows.append({
                    "BBREF_ID":         bbref_id,
                    "PLAYER_NAME":      player_name,
                    "POSITION":         get("pos"),
                    "HEIGHT":           get("height"),
                    "WEIGHT":           get("weight"),
                    "BIRTH_DATE":       get("birth_date"),
                    "YEARS_EXPERIENCE": get("years_experience"),
                    "COLLEGE":          get("college"),
                })

        except RuntimeError:
            raise
        except Exception as e:
            print(f"  ERROR {team}: {e}")

        time.sleep(5.0)

    bios_df = pl.DataFrame(all_rows).unique(subset=["BBREF_ID"], keep="first")
    bios_df.write_csv(bio_out)
    print(f"Saved {bio_out.name} — {len(bios_df):,} players")
    print(bios_df.head(3))

Scraping rosters: 100%|██████████| 30/30 [02:39<00:00,  5.32s/it]

Saved player_bios.csv — 569 players
shape: (3, 8)
┌───────────┬──────────────┬──────────┬────────┬────────┬──────────────┬─────────────┬─────────────┐
│ BBREF_ID  ┆ PLAYER_NAME  ┆ POSITION ┆ HEIGHT ┆ WEIGHT ┆ BIRTH_DATE   ┆ YEARS_EXPER ┆ COLLEGE     │
│ ---       ┆ ---          ┆ ---      ┆ ---    ┆ ---    ┆ ---          ┆ IENCE       ┆ ---         │
│ str       ┆ str          ┆ str      ┆ str    ┆ str    ┆ str          ┆ ---         ┆ str         │
│           ┆              ┆          ┆        ┆        ┆              ┆ str         ┆             │
╞═══════════╪══════════════╪══════════╪════════╪════════╪══════════════╪═════════════╪═════════════╡
│ millele01 ┆ Leonard      ┆ SF       ┆ 6-10   ┆ 220    ┆ November 26, ┆ 1           ┆             │
│           ┆ Miller       ┆          ┆        ┆        ┆ 2003         ┆             ┆             │
│ martity01 ┆ Tyrese       ┆ SG       ┆ 6-6    ┆ 215    ┆ March 7,     ┆ 1           ┆ Rhode Islan │
│           ┆ Martin       ┆          ┆  

In [11]:
bios = pl.read_csv(RAW_DIR / "player_bios.csv")
print(f"player_bios.csv: {len(bios):,} rows")
print(f"Positions: {sorted(bios['POSITION'].unique().to_list())}")
print(bios.head(5))

player_bios.csv: 569 rows
Positions: ['C', 'PF', 'PG', 'SF', 'SG']
shape: (5, 8)
┌───────────┬──────────────┬──────────┬────────┬────────┬──────────────┬─────────────┬─────────────┐
│ BBREF_ID  ┆ PLAYER_NAME  ┆ POSITION ┆ HEIGHT ┆ WEIGHT ┆ BIRTH_DATE   ┆ YEARS_EXPER ┆ COLLEGE     │
│ ---       ┆ ---          ┆ ---      ┆ ---    ┆ ---    ┆ ---          ┆ IENCE       ┆ ---         │
│ str       ┆ str          ┆ str      ┆ str    ┆ i64    ┆ str          ┆ ---         ┆ str         │
│           ┆              ┆          ┆        ┆        ┆              ┆ str         ┆             │
╞═══════════╪══════════════╪══════════╪════════╪════════╪══════════════╪═════════════╪═════════════╡
│ millele01 ┆ Leonard      ┆ SF       ┆ 6-10   ┆ 220    ┆ November 26, ┆ 1           ┆             │
│           ┆ Miller       ┆          ┆        ┆        ┆ 2003         ┆             ┆             │
│ martity01 ┆ Tyrese       ┆ SG       ┆ 6-6    ┆ 215    ┆ March 7,     ┆ 1           ┆ Rhode Islan │
│         

## 4. Summary

After running this notebook, `../data/raw/` should contain **11 files**:
- `game_logs_*.csv` × 5 (~25,000 rows each)
- `salaries_*.csv` × 5 (~450–500 rows each)
- `player_bios.csv` (1 row per unique player — position, birth date, experience, draft info)

Proceed to `02_data_wrangling.ipynb`.